# 00 - Environment check

Run this notebook **before** the workshop to make sure your environment
can reach the cloud and stream data. It takes about a minute.

If every cell runs without an error, you are ready for the two stories:

* **01** - *What changed in 30 years of German farmland?*
* **02** - *Do crops leave a fingerprint in the bare soil?*

## 1. Packages and versions

In [ ]:
import sys

import dask
import matplotlib
import numpy as np
import odc.stac
import pandas as pd
import planetary_computer
import pystac_client
import rasterio
import scipy
import xarray as xr

import eo_utils as eo

print("python       ", sys.version.split()[0])
print("odc-stac     ", odc.stac.__version__)
print("xarray       ", xr.__version__)
print("dask         ", dask.__version__)
print("rasterio     ", rasterio.__version__)
print("pystac-client", pystac_client.__version__)
print("scipy        ", scipy.__version__)
print("pandas       ", pd.__version__)
print("matplotlib   ", matplotlib.__version__)
print()
print("eo_utils loaded from:", eo.__file__)

## 2. Cloud-friendly defaults

In [ ]:
eo.apply_cloud_defaults()
print("GDAL retries + limited Dask connections configured.")

## 3. Connect to the STAC catalogues

In [ ]:
pc = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
thuenen = pystac_client.Client.open("https://eodata.thuenen.de/stac/api/v1/")

print("Planetary Computer : https://planetarycomputer.microsoft.com/api/stac/v1")
print("Thünen Institute   : https://eodata.thuenen.de/stac/api/v1/")
print()
print("Thünen collections :", [c.id for c in thuenen.get_collections()])

## 4. Stream a tiny slice of Landsat (no download)

In [ ]:
bbox = [8.98, 53.28, 9.08, 53.32]  # ~7 x 4 km, Lower Saxony

search = pc.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime="2020-06-01/2020-06-30",
    query={"eo:cloud_cover": {"lt": 20}},
)
items = planetary_computer.sign(search.item_collection())
print(f"{len(items)} Landsat scenes found")

ds = odc.stac.load(
    items,
    bands=["red", "nir08", "qa_pixel"],
    bbox=bbox,
    resolution=30,
    chunks={"time": -1, "x": 256, "y": 256},
    groupby="solar_day",
    fail_on_error=False,
)
clear = (ds["qa_pixel"] & 0b11010) == 0
red = (ds["red"] * 2.75e-5 - 0.2).where(clear)
nir = (ds["nir08"] * 2.75e-5 - 0.2).where(clear)
ndvi = ((nir - red) / (nir + red)).clip(-1, 1)

with eo.compute_with_progress():
    ndvi = ndvi.compute()

print("NDVI stack shape:", ndvi.shape)
print("Mean NDVI       :", round(float(np.nanmean(ndvi.values)), 3))

## 5. Check the Thünen legend and the GIF capability

In [ ]:
from PIL import Image  # noqa: F401  (matplotlib's GIF writer needs Pillow)

item = list(thuenen.search(collections=["hist-crop-type-map"],
                           bbox=bbox, datetime="2018-01-01/2018-12-31").items())[0]
legend = eo.legend_from_item(item)
print("Classes in the STAC legend:", len(legend))
print(legend[["title", "color_hint"]].head())

## All good?

If the cells above ran, your environment is workshop-ready.

**Troubleshooting**

* `ModuleNotFoundError` -> wrong kernel. Select the `cloud-native-eo`
  environment (Kernel -> Change Kernel).
* HTTP 429 / 503 -> rate-limited. Re-run the cell; the GDAL retry
  settings from `eo.apply_cloud_defaults()` will back off.
* No internet / corporate proxy -> STAC and cloud streaming need
  outbound HTTPS to `planetarycomputer.microsoft.com` and
  `eodata.thuenen.de`.